# Explore Lagoon Creek topo

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools

In [ ]:
topodir = './LagoonCreek_topofiles'
topo1s = topotools.Topography(f'{topodir}/LagoonCreek1s.asc', topo_type=3)
topo13s = topotools.Topography(f'{topodir}/LagoonCreek13s.asc', topo_type=3)

## Note mismatch between 1/3" topo and newer 1" topo

**This has been fixed.**  The old CRM vol 7 file was being used previously.

See [Workshop problem page on topo](https://depts.washington.edu/ptha/CopesHubTsunamis/LagoonCreek/topo/lagooncreektopo/)
for information on the source of the 1/3" Crescent City DEM fromn 2010 and the 1" CRM volume 7 from 2025.

In [ ]:
fig,ax = subplots(figsize=(8,8))
topo1s.plot(axes=ax, limits=(-20,20))
topo13s.plot(axes=ax, limits=(-20,20), add_colorbar=False)
axis(topo1s.extent);

## Make interpolating functions:

In [ ]:
topo13s_fcn = topo13s.make_function()
topo1s_fcn = topo1s.make_function()

In [ ]:
xt1,yt1, xt2,yt2 = [-124.16, 41.599, -124.09, 41.599]
npts = 1000
xtrans = linspace(xt1,xt2,npts)
ytrans = linspace(yt1,yt2,npts)
Btrans1s = topo1s_fcn(xtrans, ytrans)
Btrans13s = topo13s_fcn(xtrans, ytrans)

figure(figsize=(8,5))
plot(xtrans, Btrans1s, 'g', label='topo1s data at 1"')
plot(xtrans, Btrans13s, 'r', label='topo13s data at 1/3"')
grid(True)
legend(framealpha=1)
ylim(-50,50)
title(f'Transect of data at y = {yt1:.4f}');

## Transects of 1/3" topo in Lagoon Creek

In [ ]:
trA = [-124.1040, -124.0958, 41.5964, 41.5918]
trB = [-124.1025, -124.0966, 41.5900, 41.5913]
fig,ax = subplots(figsize=(7,7))
topo13s.plot(axes=ax, limits=(-20,20))
ax.set_title('Topography and transects')

tr = trA
ax.plot(tr[:2], tr[2:], 'k')
ax.text(tr[0],tr[2]+0.0002, 'A')
ax.text(tr[1],tr[3]+0.0002, "A'")
tr = trB
ax.plot(tr[:2], tr[2:], 'k')
ax.text(tr[0],tr[2]+0.0002, 'B')
ax.text(tr[1],tr[3]-0.0002, "B'")

fig,ax = subplots(figsize=(7,4))
ax.set_title("Topography on transect A to A'")
tr = trA
xtr = linspace(tr[0], tr[1], 1000)
ytr = linspace(tr[2], tr[3], 1000)
Btr = topo13s_fcn(xtr, ytr)
ticklabel_format(useOffset=False)
ax.plot(xtr, Btr, 'g')
ax.set_xlabel('longitude')
ax.set_ylabel('meters above MHW')
ax.grid(True)

fig,ax = subplots(figsize=(7,4))
ax.set_title("Topography on transect B to B'")
tr = trB
xtr = linspace(tr[0], tr[1], 1000)
ytr = linspace(tr[2], tr[3], 1000)
Btr = topo13s_fcn(xtr, ytr)
ticklabel_format(useOffset=False)
ax.plot(xtr, Btr, 'g')
ax.set_xlabel('longitude')
ax.set_ylabel('meters above MHW')
ax.grid(True);


## Topo from Ignacio

For comparison.

In [ ]:
file_path = '/Users/rjl/Downloads/exportImage_fine.tiff'
import rasterio

print(f'rasterio data for {file_path}:\n')
with rasterio.open(file_path) as src:
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Number of bands: {src.count}")
    print(f"Width/Height: {src.width}x{src.height}")
    
    meta = src.meta
    print('\nsrc.meta = \n')
    for k in meta.keys():
        print(f'{k:<30}{meta[k]}')
    
    tags = src.tags()
    print('\nsrc.tags = \n')
    for k in tags.keys():
        print(f'{k:<30}:  {tags[k]}')

    x1,y1,x2,y2 = array(src.bounds)
    print(f'\nLeft and right bounds: {x1:.9f} ... {x2:.9f}')
    print(f'Bottom and top bounds: {y1:.9f} ... {y2:.9f}')
    nx = (x2 - x1) * 3 * 3600
    print(f'Distance between left,right bounds: {nx} 1/3 arcsec cells')
    ny = (y2 - y1) * 3 * 3600
    print(f'Distance between bottom,top bounds: {ny} 1/3 arcsec cells')

    topotif = src.read()
    bounds = src.bounds

In [ ]:
Z = flipud(topotif[0,:,:])
x1,x2,y1,y2 = extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
dx = (x2-x1)/Z.shape[0]
dy = (y2-y1)/Z.shape[1]
print(f'dx = {dx:.7f} = {dx*3600:.7f} arcseconds')
print(f'dy = {dy:.7f} = {dy*3600:.7f} arcseconds')

x = arange(x1+0.5*dx, x2, dx)
y = arange(y1+0.5*dy, y2, dy)

topo_from_tif = topotools.Topography()
topo_from_tif.set_xyZ(x,y,Z)

topo_from_tif_fcn = topo_from_tif.make_function()

In [ ]:
topo_from_tif.crop(coarsen=3).plot()

## Check topo at gauges:

In [ ]:
# Gauge 3

xg = -124.0954
yg = 41.5891
Bg = topo13s_fcn(xg, yg)
print(f'Initial topo at gauge is {Bg:.3f} m relative to MHW from 1/3 arcsec CC DEM')
Bg = topo_from_tif_fcn(xg, yg)
print(f'Initial topo at gauge is {Bg:.3f} m relative to MHW from topo_tif')

## Download CRM volume 7 again

In [ ]:
#url = 'https://www.ngdc.noaa.gov/thredds/dodsC/crm/crm_vol7.nc'  # WRONG!
url = 'https://www.ngdc.noaa.gov/thredds/dodsC/crm/cudem/crm_vol7_2025.nc'

extent = [-124.3, -124., 41.5, 41.8]
topo1s = topotools.read_netcdf(url, extent=extent)
print(f'Shape: (number of lat, lon values): {topo1s.Z.shape}')

In [ ]:
topo1s_fcn = topo1s.make_function()

In [ ]:
xt1,yt1, xt2,yt2 = [-124.16, 41.599, -124.09, 41.599]
npts = 1000
xtrans = linspace(xt1,xt2,npts)
ytrans = linspace(yt1,yt2,npts)
Btrans1s = topo1s_fcn(xtrans, ytrans)
Btrans13s = topo13s_fcn(xtrans, ytrans)
Btrans_from_tif = topo_from_tif_fcn(xtrans, ytrans)

figure(figsize=(8,5))
plot(xtrans, Btrans1s, 'g', label='topo1s data at 1" (CRM vol 7 2025)')
plot(xtrans, Btrans_from_tif, 'b', label='topo_from_tif at 1/3 (Global Mosaic DEM)"')
plot(xtrans, Btrans13s, 'r', label='topo13s at 1/3" (Crescent City DEM 2010)')

grid(True)
legend(framealpha=1)
ylim(-50,50)
title(f'Transect of data at y = {yt1:.4f}');
if 1:
    fname = 'Transect_comparison_y41p5990.png'
    savefig(fname)
    print('Created ',fname)